# EE-411 Lottery Ticket Hypothesis -  Experiment : Random Pruning and Iterative Pruning

**Author:** Xavier  
**Model:** Conv6
**Dataset:** CIFAR-10  
**Date:** 17/01/2026

---

## Instructions

1. **Define your model** in the "Model Definition" section
2. **Train baseline model** to get initial performance
3. **Run pruning experiments** using the shared pruning algorithms
4. **Visualize and analyze** your results
5. **Save your results** to the `results/` folder

## Quick Links

- [Paper](https://arxiv.org/abs/1803.03635): "The Lottery Ticket Hypothesis"
- [Results](../../results/): Save checkpoints and figures here



## 1. Setup and Imports

In [1]:
# To save later on the results in my driive

from google.colab import drive
drive.mount("/content/drive")



Mounted at /content/drive


In [2]:
# To see where is located my notebook
import os
print(os.getcwd())

/content


In [3]:
!ls /content/drive/MyDrive/ee411-lottery-ticket-hypothesis/results

checkpoints  conv6_results.json  conv6_results_random.json  figures


In [4]:
import os
os.makedirs("/content/drive/MyDrive/ee411-lottery-ticket-hypothesis/results", exist_ok=True)

In [5]:
# Let's import the useful libraries

import random
import numpy as np
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torch.nn.functional as F
import time
import copy
import torch
import json, os
import matplotlib.pyplot as plt

# Fix all random seeds
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# For full determinism
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Import the best device available
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [6]:
RESULTS_PATH = "/content/drive/MyDrive/ee411-lottery-ticket-hypothesis/results/conv6_results.json"

RESULTS_PATH_RANDOM = "/content/drive/MyDrive/ee411-lottery-ticket-hypothesis/results/conv6_results_random.json"
def load_results(path=RESULTS_PATH):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {}  # vide si rien

def save_results(results, path=RESULTS_PATH):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(results, f, indent=2)
    os.replace(tmp, path)  # écriture “safe”

In [7]:
def _to_cpu_state_dict(sd):
    return {k: v.detach().cpu() for k, v in sd.items()}

def _to_cpu_masks(masks):
    return {k: v.detach().cpu() for k, v in masks.items()}

def _rng_state():
    st = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.random.get_rng_state(),
    }
    if torch.cuda.is_available():
        st["torch_cuda"] = torch.cuda.get_rng_state_all()
    return st

def _set_rng_state(st):
    random.setstate(st["python"])
    np.random.set_state(st["numpy"])
    torch.random.set_rng_state(st["torch"])
    if torch.cuda.is_available() and "torch_cuda" in st:
        torch.cuda.set_rng_state_all(st["torch_cuda"])

def save_ckpt(path, payload: dict):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    payload = dict(payload)
    payload["timestamp"] = time.time()
    torch.save(payload, tmp)
    os.replace(tmp, path)

def load_ckpt(path, map_location="cpu"):
    return torch.load(path, map_location=map_location)

## Data Loading

In [8]:
# Hyperparameters
BATCH_SIZE = 60
TEST_BATCH_SIZE = 512
LEARNING_RATE = 3e-4

In [9]:
# load the train dataset

# Define transforms
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

test_dataset = torchvision.datasets.CIFAR10(
    root='/content/drive/MyDrive/ee411-lottery-ticket-hypothesis/data',
    train=False,
    download=True,
    transform=transform_test)


train_dataset = torchvision.datasets.CIFAR10(
    root='/content/drive/MyDrive/ee411-lottery-ticket-hypothesis/data',
    train=True,
    download=True,
    transform=transform_train)

# Split the dataset into 45k-5k samples for training-validation.
from torch.utils.data import random_split
train_dataset,  valid_dataset = random_split(
    train_dataset,
    lengths=[45000, 5000],
    generator=torch.Generator().manual_seed(42) # we use a generator to insure reproducibilty
)


# Create data loaders
batch_size = 60  # From paper

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2)

valid_dataloader = DataLoader(
    dataset=valid_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=2)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=2)

print(f"Train: {len(train_dataset)} samples")
print(f"Val:   {len(valid_dataset)} samples")
print(f"Test:  {len(test_dataset)} samples")
print(f"Batches per epoch: {len(train_dataloader)}")




Train: 45000 samples
Val:   5000 samples
Test:  10000 samples
Batches per epoch: 750


## 3. Model Definition

**TODO: Define your model here**

In [10]:
def count_parameters(model: nn.Module, only_trainable: bool = True) -> int:
    """
    Count parameters in a PyTorch model.

    only_trainable=True counts only parameters with requires_grad=True.
    """
    if only_trainable:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())

In [11]:
class Conv6(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # Conv-6: (64,64,pool) -> (128,128,pool) -> (256,256,pool)
        # All convs are 3x3 with padding=1 to preserve spatial size within each block.

        # CIFAR-10 input is (3, 32, 32)
        # After 3 pools: 32 -> 16 -> 8 -> 4, so final map is (256, 4, 4)
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )

        # Flatten size is 256*4*4 = 4096 for CIFAR-10
        self.classifier = nn.Sequential(
            nn.Flatten(1),
            nn.Linear(256 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Create model instance
model = Conv6().to(device)

# Print model info
total_params = count_parameters(model)
print(f"Model: Conv6")
print(f"Total parameters: {total_params:,}")
print(f"\nModel architecture:")
print(model)

Model: Conv6
Total parameters: 2,262,602

Model architecture:
Conv6(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequ

Regarding the architecture of Conv6 here is some justifications :
  
-Using 3×3 matches the paper statement and gives a standard receptive-field growth.

-Using padding=1 ensures only pooling changes resolution; this matches the “64,64,pool …” style description (conv pairs keep size, pool halves it).

-Removing adaptive pooling and using flatten + FC matches the listed FC layers (256,256,10) setup.

-LazyLinear(256) avoids hardcoding the flatten dimension while keeping the paper’s FC widths

### Defining Functions : Iterative pruning , Random Pruning, Evaluation &  Training

In [12]:
#Prunable parameters, mask application, and counting remaining weights

def iter_prunable_params(model: nn.Module):
    # prune conv/linear weights only (exclude biases and BN)
    for name, p in model.named_parameters():
        if p.requires_grad and name.endswith(".weight") and p.dim() >= 2:
            yield name, p

@torch.no_grad()
def apply_masks(model, masks):
    for name, p in model.named_parameters():
        if name in masks:
            p.mul_(masks[name])

def masks_keep_ratio(masks):
    kept = sum(int(m.sum().item()) for m in masks.values())
    total = sum(m.numel() for m in masks.values())
    return kept / total


In [13]:
#Random mask (global)
@torch.no_grad()
def make_global_random_masks(model, keep_ratio: float, seed: int):
    params = list(iter_prunable_params(model))
    device = params[0][1].device
    total = sum(p.numel() for _, p in params)
    k = max(1, int(keep_ratio * total))

    g = torch.Generator(device=device).manual_seed(seed)
    idx = torch.randperm(total, generator=g, device=device)[:k]

    masks = {}
    offset = 0
    for name, p in params:
        n = p.numel()
        mask_flat = torch.zeros(n, device=device, dtype=p.dtype)
        local = idx[(idx >= offset) & (idx < offset + n)] - offset
        mask_flat[local] = 1.0
        masks[name] = mask_flat.view_as(p)
        offset += n
    return masks

In [14]:
# Magnitude pruning “by group” (conv vs fc) per round (paper-style)

def is_conv_weight(name, param):
    return param.dim() == 4  # Conv2d weight: (out,in,k,k)

def is_fc_weight(name, param):
    return param.dim() == 2  # Linear weight: (out,in)

@torch.no_grad()
def prune_by_magnitude_inplace(model, masks, conv_prune_frac=0.15, fc_prune_frac=0.20):
    # prune additional fraction among CURRENTLY-KEPT weights
    for name, p in iter_prunable_params(model):
        w = p.detach()
        m = masks[name].detach()

        # consider only currently kept weights
        kept_vals = w.abs()[m.bool()]
        if kept_vals.numel() == 0:
            continue

        prune_frac = conv_prune_frac if is_conv_weight(name, p) else (fc_prune_frac if is_fc_weight(name, p) else 0.0)
        if prune_frac <= 0:
            continue

        k_prune = int(prune_frac * kept_vals.numel())
        if k_prune < 1:
            continue

        # threshold: prune the smallest magnitudes among kept weights
        thresh = torch.kthvalue(kept_vals, k_prune).values
        to_prune = (w.abs() <= thresh) & (m > 0)

        new_m = m.clone()
        new_m[to_prune] = 0.0
        masks[name] = new_m.to(p.dtype)

    # enforce immediately
    apply_masks(model, masks)
    return masks

In [15]:
# Training that returns “early-stop iteration” = argmin val loss


@torch.no_grad()
def evaluate_model(model, loader, device, criterion):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        bs = x.size(0)
        total_loss += loss.item() * bs
        total_correct += (logits.argmax(1) == y).sum().item()
        total += bs
    return total_loss / total, total_correct / total

def train_masked_and_get_best_iter(model, train_loader, val_loader, test_loader, device,
                                  optimizer, max_steps, masks=None):
    criterion = nn.CrossEntropyLoss()
    model.to(device)
    if masks is not None:
        apply_masks(model, masks)

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    best_iter = 0

    step = 0
    model.train()
    train_iter = iter(train_loader)

    t0 = time.time()
    t_last = t0
    last_step = 0

    while step < max_steps:
        if (step % 1000 == 0) and (step != 0):
            now = time.time()
            dt = now - t_last
            dsteps = step - last_step
            ms_per_step = 1000.0 * dt / max(1, dsteps)
            total_min = (now - t0) / 60.0
            print(f"[PROGRESS] step {step}/{max_steps} | {dt:.1f}s for last {dsteps} steps "
                  f"(~{ms_per_step:.2f} ms/step) | total {total_min:.1f} min")
            t_last = now
            last_step = step


        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        if masks is not None:
            apply_masks(model, masks)

        step += 1

        if step % 50 == 0 or step == max_steps:
            vloss, _ = evaluate_model(model, val_loader, device, criterion)
            if vloss < best_val_loss:
                best_val_loss = vloss
                best_iter = step
                best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    _, test_acc = evaluate_model(model, test_loader, device, nn.CrossEntropyLoss())
    return best_iter, test_acc

### Random Pruning and Iterative Pruning Functions

In [16]:
def run_random_curve_resume(
    model_ctor, keep_ratios, trials,
    train_loader, val_loader, test_loader,
    device, max_steps, lr,
    results_path=RESULTS_PATH_RANDOM,
    resume: bool = True,
    overwrite: bool = False,
    seed_base: int = 1000
):
    if resume:
        results = load_results(results_path)
        print(f"[LOAD] Loaded existing results from: {results_path}")
    else:
        results = {}
        print("[NEW RUN] Not resuming from disk (starting fresh).")

    results.setdefault("random", {})

    if overwrite:
        results["random"] = {}
        print("[OVERWRITE] Wiped previous 'random' results in memory.")

    for kr in keep_ratios:

        print("\n" + "-" * 70)
        print(f"keep_ratio : {kr:.3f} meaning {kr*100:.1f}% weights remaining")

        # keep your budget logic exactly
        if kr > 0.4:
            cur_steps = max_steps
            cur_trials = trials
        elif kr < 0.4 and kr > 0.15:
            cur_steps = 15000
            cur_trials = 4
        else:
            cur_steps = 25000
            cur_trials = 4

        print(f"[BUDGET] max_steps={cur_steps} | trials={cur_trials} | lr={lr} | device={device}")

        for t in range(cur_trials):
            key = f"kr={kr:.6f}_trial={t}"

            if resume and key in results["random"]:
                print(f"[SKIP] {key} already computed")
                continue

            print(f"\n[TRIAL] {t+1}/{cur_trials} (seed={seed_base+t})")
            print(f"[RUN] random keep_ratio={kr:.3f} trial {t+1}/{cur_trials}")

            model = model_ctor().to(device)
            masks = make_global_random_masks(model, keep_ratio=kr, seed=seed_base + t)
            opt = torch.optim.Adam(model.parameters(), lr=lr)

            best_iter, test_acc = train_masked_and_get_best_iter(
                model, train_loader, val_loader, test_loader, device,
                optimizer=opt, max_steps=cur_steps, masks=masks
            )

            print(f"[TRIAL DONE] best_iter={best_iter} ({best_iter/1000:.1f}K) | test_acc@best={test_acc:.4f}")

            results["random"][key] = {
                "keep_ratio": float(kr),
                "trial": int(t),
                "best_iter": int(best_iter),
                "test_acc": float(test_acc),
                "max_steps": int(cur_steps),
                "lr": float(lr),
                "seed": int(seed_base + t),
            }
            save_results(results, results_path)
            print(f"[SAVE] wrote results for {key} to {results_path}")

    print("\n[FINISHED] random curve runs completed.")
    return results

In [17]:
def run_ticket_curve_resume(
    model_ctor, keep_ratios, trials,
    train_loader, val_loader, test_loader,
    device, max_steps, lr, conv_prune=0.15, fc_prune=0.20,
    results_path=RESULTS_PATH,
    resume: bool = True,
    overwrite: bool = False,
):
    if resume:
        results = load_results(results_path)
    else:
        results = {}

    results.setdefault("ticket", {})
    if overwrite:
        results["ticket"] = {}

    for kr_target in keep_ratios:

        # keep your budget logic
        if kr_target > 0.4:
            cur_steps = max_steps
        elif kr_target < 0.4 and kr_target > 0.15:
            cur_steps = 12000
        else:
            cur_steps = 15000

        print(f"\n[TICKET] keep_ratio={kr_target:.3f} | steps={cur_steps} | trials={trials} "
              f"| conv_prune={conv_prune} fc_prune={fc_prune}")

        for t in range(trials):
            key = f"kr={kr_target:.6f}_trial={t}"

            if resume and key in results["ticket"]:
                print(f"  [SKIP] trial {t+1}/{trials} already done ({key})")
                continue

            print(f"  [TRIAL {t+1}/{trials}] starting...")

            # init model and save W0
            model0 = model_ctor().to(device)
            init_state = {k: v.detach().clone() for k, v in model0.state_dict().items()}
            masks = {name: torch.ones_like(p) for name, p in iter_prunable_params(model0)}

            current_keep = masks_keep_ratio(masks)
            model = model_ctor().to(device)

            rounds = 0
            while current_keep > kr_target:
                rounds += 1

                model.load_state_dict(init_state)
                apply_masks(model, masks)

                opt = torch.optim.Adam(model.parameters(), lr=lr)
                _best_iter, _test_acc = train_masked_and_get_best_iter(
                    model, train_loader, val_loader, test_loader, device,
                    optimizer=opt, max_steps=cur_steps, masks=masks
                )

                masks = prune_by_magnitude_inplace(model, masks, conv_prune, fc_prune)
                current_keep = masks_keep_ratio(masks)

                # useful progress print: pruning rounds + keep ratio progress
                print(f"    round {rounds}: keep_ratio -> {current_keep:.3f}")

                if current_keep <= kr_target:
                    break

            # final ticket train/eval
            model.load_state_dict(init_state)
            apply_masks(model, masks)
            opt = torch.optim.Adam(model.parameters(), lr=lr)

            best_iter, test_acc = train_masked_and_get_best_iter(
                model, train_loader, val_loader, test_loader, device,
                optimizer=opt, max_steps=cur_steps, masks=masks
            )

            print(f"  [TRIAL {t+1}/{trials}] done | rounds={rounds} | best_iter={best_iter} "
                  f"({best_iter/1000:.1f}K) | test_acc@best={test_acc:.4f}")

            results["ticket"][key] = {
                "keep_ratio": float(kr_target),
                "trial": int(t),
                "rounds": int(rounds),
                "best_iter": int(best_iter),
                "test_acc": float(test_acc),
                "max_steps": int(cur_steps),
                "lr": float(lr),
                "conv_prune": float(conv_prune),
                "fc_prune": float(fc_prune),
            }

            save_results(results, results_path)

    return results

## Plotting Function

In [18]:
# Plotting
def plot_figure(random_res, ticket_res):
    # This function plots the random and winning ticket curves.
    # random_res/ticket_res are lists of (kr, iters_tensor, accs_tensor)
    x = np.array([kr*100 for kr,_,_ in random_res])

    # left: best_iter (K)
    plt.figure()
    for label, res, ls in [("random", random_res, "--"), ("Conv-6 ticket", ticket_res, "-")]:
        mean_iters = np.array([r[1].mean().item() for r in res]) / 1000.0
        std_iters  = np.array([r[1].std(unbiased=False).item() for r in res]) / 1000.0
        plt.errorbar(x, mean_iters, yerr=std_iters, linestyle=ls, marker="o", capsize=3, label=label)
    plt.xlabel("Percent of Weights Remaining")
    plt.ylabel("Iteration of Minimum Validation Loss (K)")
    plt.gca().invert_xaxis()
    plt.legend()
    plt.show()

    # right: test accuracy at that iteration
    plt.figure()
    for label, res, ls in [("random", random_res, "--"), ("Conv-6 ticket", ticket_res, "-")]:
        mean_acc = np.array([r[2].mean().item() for r in res])
        std_acc  = np.array([r[2].std(unbiased=False).item() for r in res])
        plt.errorbar(x, mean_acc, yerr=std_acc, linestyle=ls, marker="o", capsize=3, label=label)
    plt.xlabel("Percent of Weights Remaining")
    plt.ylabel("Test Accuracy at Best-Validation Iteration")
    plt.gca().invert_xaxis()
    plt.legend()
    plt.show()

In [19]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Running Experiments

In [20]:
# =========================
#  Experiment settings
# =========================
model_ctor = lambda: Conv6()

keep_ratios = [1.0, 0.71, 0.412, 0.170, 0.071, 0.030, 0.013]  # x-axis points (weights remaining)

max_steps = 10000    # paper: 30K iterations

random_trials = 4   # dashed lines in caption
ticket_trials = 2    # solid lines in caption

# pruning per round for Conv-6 (from table): conv15% fc20%
conv_prune_frac = 0.15
fc_prune_frac   = 0.20

In [21]:
%%time
# =========================
# Run random subnetworks (dashed)
# =========================
set_seed(123)
random_res = run_random_curve_resume(
    model_ctor=model_ctor,
    keep_ratios=keep_ratios,
    trials=random_trials,
    train_loader=train_dataloader,
    val_loader=valid_dataloader,
    test_loader=test_dataloader,
    device=device,
    max_steps=max_steps,
    lr=LEARNING_RATE,
)

[LOAD] Loaded existing results from: /content/drive/MyDrive/ee411-lottery-ticket-hypothesis/results/conv6_results_random.json

----------------------------------------------------------------------
keep_ratio : 1.000 meaning 100.0% weights remaining
[BUDGET] max_steps=10000 | trials=4 | lr=0.0003 | device=cuda
[SKIP] kr=1.000000_trial=0 already computed
[SKIP] kr=1.000000_trial=1 already computed
[SKIP] kr=1.000000_trial=2 already computed
[SKIP] kr=1.000000_trial=3 already computed

----------------------------------------------------------------------
keep_ratio : 0.710 meaning 71.0% weights remaining
[BUDGET] max_steps=10000 | trials=4 | lr=0.0003 | device=cuda
[SKIP] kr=0.710000_trial=0 already computed
[SKIP] kr=0.710000_trial=1 already computed
[SKIP] kr=0.710000_trial=2 already computed
[SKIP] kr=0.710000_trial=3 already computed

----------------------------------------------------------------------
keep_ratio : 0.412 meaning 41.2% weights remaining
[BUDGET] max_steps=10000 | t

In [ ]:
%%time
# =========================
# Run winning tickets (solid)
# =========================
set_seed(456)
ticket_res = run_ticket_curve_resume(
    model_ctor=model_ctor,           # e.g. lambda: Conv6()
    keep_ratios=keep_ratios,         # list of sparsity targets
    trials=ticket_trials,            # number of trials per keep_ratio (except <0.15 -> forced to 2)
    train_loader=train_dataloader,
    val_loader=valid_dataloader,
    test_loader=test_dataloader,
    device=device,                   # "cuda" or "cpu"
    max_steps=max_steps,             # unused as-is (local_max_steps overrides inside)
    lr=LEARNING_RATE,
    conv_prune=conv_prune_frac,
    fc_prune=fc_prune_frac,
)



[TICKET] keep_ratio=1.000 | steps=10000 | trials=2 | conv_prune=0.15 fc_prune=0.2
  [SKIP] trial 1/2 already done (kr=1.000000_trial=0)
  [SKIP] trial 2/2 already done (kr=1.000000_trial=1)

[TICKET] keep_ratio=0.710 | steps=10000 | trials=2 | conv_prune=0.15 fc_prune=0.2
  [SKIP] trial 1/2 already done (kr=0.710000_trial=0)
  [SKIP] trial 2/2 already done (kr=0.710000_trial=1)

[TICKET] keep_ratio=0.412 | steps=10000 | trials=2 | conv_prune=0.15 fc_prune=0.2
  [SKIP] trial 1/2 already done (kr=0.412000_trial=0)
  [SKIP] trial 2/2 already done (kr=0.412000_trial=1)

[TICKET] keep_ratio=0.170 | steps=12000 | trials=2 | conv_prune=0.15 fc_prune=0.2
  [SKIP] trial 1/2 already done (kr=0.170000_trial=0)
  [SKIP] trial 2/2 already done (kr=0.170000_trial=1)

[TICKET] keep_ratio=0.071 | steps=15000 | trials=2 | conv_prune=0.15 fc_prune=0.2
  [SKIP] trial 1/2 already done (kr=0.071000_trial=0)
  [SKIP] trial 2/2 already done (kr=0.071000_trial=1)

[TICKET] keep_ratio=0.030 | steps=15000 | tr

In [ ]:
def process_results_for_plot(raw_results_dict):
    processed = []
    grouped_by_kr = {}
    for key, val in raw_results_dict.items():
        kr = val["keep_ratio"]
        if kr not in grouped_by_kr:
            grouped_by_kr[kr] = {"best_iters": [], "test_accs": []}
        grouped_by_kr[kr]["best_iters"].append(val["best_iter"])
        grouped_by_kr[kr]["test_accs"].append(val["test_acc"])

    for kr in sorted(grouped_by_kr.keys(), reverse=True):
        iters = torch.tensor(grouped_by_kr[kr]["best_iters"], dtype=torch.float32)
        accs = torch.tensor(grouped_by_kr[kr]["test_accs"], dtype=torch.float32)
        processed.append((kr, iters, accs))
    return processed

# =========================
# 5) Plot Figure 1 (two panels)
# =========================

# Process the raw results dictionaries into the format expected by plot_figure
processed_random_res = process_results_for_plot(random_res["random"])
processed_ticket_res = process_results_for_plot(ticket_res["ticket"])

plot_figure(processed_random_res, processed_ticket_res)


NameError: name 'random_res' is not defined

In [ ]:
# =========================
# 6) Optional: print raw results
# =========================
print("RANDOM:")
for kr, iters, accs in processed_random_res:
    print(f"keep={kr:.3f} | best_iter mean={iters.mean():.1f} std={iters.std(unbiased=False):.1f} | "
          f"test_acc mean={accs.mean():.4f} std={accs.std(unbiased=False):.4f}")

print("\nTICKET:")
for kr, iters, accs in processed_ticket_res:
    print(f"keep={kr:.3f} | best_iter mean={iters.mean():.1f} std={iters.std(unbiased=False):.1f} | "
          f"test_acc mean={accs.mean():.4f} std={accs.std(unbiased=False):.4f}")